# Onboarding: The Completed-Cycle Marketing System (`src_2`)

**Who this is for:** someone brand new — you don't know the business, the data, or
the code. By the end you'll understand *why this system exists*, *what decision it
helps a human make*, and *exactly how the code turns raw data into that decision*.

**How to read it:**
- **Text cells** explain the business and the idea in plain language.
- **Code cells** are the *actual system* running on real sample data — each one is
  labelled with the **business question** it answers and **what to look at**.
- **Charts** are there to build intuition, not to be pretty.

Everything runs on the bundled sample data. Only one step (the AI narration) calls
an external model and needs a key — it's clearly marked and skips gracefully.


---
# Part 1 — The business problem (no code yet)

Imagine a retailer that sells through **WhatsApp**. To get customers, it runs
**ads on Meta (Facebook/Instagram)**. The journey is:

> **Someone sees an ad → taps it → starts a WhatsApp chat → (maybe) places an order → (maybe) it gets delivered.**

Money is spent at the start (ads); value appears at the end (delivered orders).

Every so often a **cycle** ends (a block of advertising activity), and a marketing
manager has to answer three very practical questions **for each campaign**:

1. **Did it do its job?** (a brand-awareness campaign is judged differently from a
   sales campaign)
2. **What should we do next cycle?** — *scale it up*, *keep testing it*, or *stop
   funding it*?
3. **How should we split next cycle's budget** across the campaigns?

### Why this is hard
The data is **incomplete and doesn't line up**. Meta reports huge numbers of
"conversations started," but only a small slice can be matched to real WhatsApp
orders. So any honest answer must say *how confident* it is, and must **not**
pretend a shaky signal is a sure thing.

### What this system does
It takes **one completed cycle** of raw data and produces:
- **Scorecards** — clean, trustworthy performance numbers per campaign.
- **Decisions** — a transparent *scale / keep-as-test / do-not-fund* call per campaign.
- **An illustrative budget** — how 100 "units" could be split next cycle.
- **A plain-English report** — so a non-marketer can understand it.

### The one golden rule (remember this)
> **Fixed rules make the decisions. The AI only explains them.**

The numbers and the fund/stop calls come from transparent, repeatable rules you can
audit. An AI writes the story around them but can never change a number or a
decision. (And the budget is explicitly *illustrative* — a starting point for a
human, not an instruction to spend.)


---
# Part 2 — The vocabulary (still no code)

A few terms you'll see everywhere.

### The ad hierarchy
```
Campaign        ── the objective + the budget  (e.g. "Eid Gifting Premium")
  └── Adset     ── who to show it to (the audience) + delivery settings
       └── Ad   ── the thing people actually see
            └── Creative ── the specific image/message an ad uses
```

### WhatsApp outcomes (what happened after the chat started)
delivered order · refunded · cancelled · "ghosted" (chat went quiet) · negative.
Only **delivered** orders create real net revenue.

### KPIs (performance measures) in plain words
| KPI | Plain meaning |
|---|---|
| **ROAS** (net) | net revenue ÷ ad spend — "how many EGP back per EGP spent" |
| **AOV** | average value of a delivered order |
| **Delivered rate** | of the chats we observed, how many became delivered orders |
| **Negative-outcome rate** | how often chats went badly (a risk guardrail) |

### Campaign *types* — each judged by its own yardstick
There are 8 types (awareness, always-on, promotional, seasonal, launch, scale,
retention, experimental). **Each type has one *primary KPI* that decides whether it
succeeded** — you don't judge an awareness campaign on sales, or a sales campaign on
reach.

### The three questions the system keeps separate
1. **Target achievement** — did it beat the benchmark on its *primary* KPI?
2. **Business outcome** — did the WhatsApp economics actually support funding it again?
3. **Evidence readiness** — is there *enough* trustworthy data to act on?

### The four possible next-cycle decisions
- **Scale** — do more of this.
- **Keep as test** — promising, but only enough evidence to keep experimenting.
- **Do not fund** — stop.
- **Insufficient evidence** — we can't responsibly decide yet.

That's the whole mental model. Now let's watch the code produce exactly these.


---
# Part 3 — Setup, then look at the raw material

In [1]:
# --- Setup (safe to ignore the details) --------------------------------------
import sys, os, json
from pathlib import Path
here = Path.cwd()
for _p in [here, *here.parents]:
    if (_p / "src_2" / "__init__.py").exists():
        ROOT = _p
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        break
import pandas as pd
import altair as alt
alt.renderers.enable("mimetype")
pd.set_option("display.max_columns", 30)
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("Ready. AI narration available:", bool(os.getenv("OPENAI_API_KEY")))


Ready. AI narration available: True


The whole input is **three JSON files** for one cycle:

| File | What it is |
|---|---|
| `meta_data.json` | the Meta ad hierarchy + daily delivery numbers |
| `conversations.json` | the WhatsApp chats and how they ended |
| `products.json` | the product catalogue |

**Business question:** *what raw material are we even working with?*

In [2]:
from src_2.ingestion import load_sample2
from src_2.paths import INPUT_DIR

raw = load_sample2(INPUT_DIR)
print("WhatsApp conversations on file:", len(raw.conversations))
print("Products on file               :", len(raw.products))
print("\nOne WhatsApp conversation (the kind of record we start from):")
one = raw.conversations[0]
print(json.dumps({k: one[k] for k in list(one)[:10]}, indent=2, default=str))


WhatsApp conversations on file: 788
Products on file               : 105

One WhatsApp conversation (the kind of record we start from):
{
  "id": "conv_001",
  "started_at": "2026-03-06T20:47:04Z",
  "last_message_at": "2026-03-06T21:00:04Z",
  "status": "closed",
  "language": "ar",
  "cycle": 2,
  "customer": {
    "id": "cust_001",
    "first_name": "Adib",
    "last_name": "Tarek-Wagdy",
    "phone": "+201147XXXXX18",
    "country": "EG",
    "first_seen_at": "2026-03-06T20:44:04Z"
  },
  "source": {
    "platform": "meta_ctwa",
    "ctwa_clid": "ARsP5vQ7xKnHmRdFcLjBcXdYjAvEz",
    "ad_id": "120209876543210017",
    "campaign_id": "120209876543220006",
    "creative_id": "120209876543230007",
    "headline": "Iftar Premium Bundle - perfect for hosting"
  },
  "messages": [
    {
      "direction": "inbound",
      "text": "\u0623\u0647\u0644\u0627\u064b. \u0639\u0627\u0648\u0632 Iftar Bundle \u0644\u0644\u0639\u0632\u0648\u0645\u0629 \u0628\u0643\u0631\u0647",
      "sent_at": "202

---
# Part 4 — The workflow, one business question at a time

The system is a **pipeline**: each step takes the previous step's output and answers
one more question. We'll run them in order.

## Step 1 — "Can we even trust this data?" 🧹

First the code **cleans** the raw files into tidy tables and **removes personal
information** (names, phone numbers, message text never travel further). Then it does
the most important honesty check: it compares how many conversations **Meta claims**
started against how many we can **actually see** in the WhatsApp data.

In [3]:
from src_2.ingestion import normalize_cycle, build_data_quality_report

canonical = normalize_cycle(raw)          # clean, privacy-safe tables
quality = build_data_quality_report(canonical)

print("Meta says conversations started :", f"{quality.meta_conversation_starts:,}")
print("We can actually observe         :", f"{quality.observed_meta_whatsapp_conversations:,}")
print("Reconciliation (observed / Meta):", f"{quality.reconciliation_ratio:.2%}")
print("Evidence verdict                :", quality.status.value)


Meta says conversations started : 116,098
We can actually observe         : 617
Reconciliation (observed / Meta): 0.53%
Evidence verdict                : limited_evidence


**What to look at:** only a tiny fraction reconciles. That's why every downstream
answer is treated as **"limited evidence"** — describing the *observed sample*, not
the whole truth. The chart makes the gap obvious:

In [4]:
recon = pd.DataFrame({
    "Source": ["Meta-attributed starts", "Observed in WhatsApp data"],
    "Conversations": [quality.meta_conversation_starts, quality.observed_meta_whatsapp_conversations],
})
alt.Chart(recon).mark_bar().encode(
    x=alt.X("Conversations:Q", title="Conversations"),
    y=alt.Y("Source:N", title=None, sort="-x"),
    color=alt.Color("Source:N", legend=None, scale=alt.Scale(range=["#B8BCC4", "#3D8DFF"])),
    tooltip=["Source:N", alt.Tooltip("Conversations:Q", format=",")],
).properties(height=120, title="Why the data is 'limited evidence': the reconciliation gap")


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Step 2 — "How did each campaign actually perform?" 📊

The code aggregates everything into **scorecards** and computes the KPIs. Two ways to
see it: the **outcome funnel** (how chats turn into delivered orders) and a
**portfolio map** (spend vs. money back).

In [5]:
from src_2.analytics import build_scorecards

scorecards = build_scorecards(canonical)
campaigns = scorecards.campaign

funnel = pd.DataFrame({
    "Stage": ["Observed conversations", "Orders created", "Delivered orders"],
    "Count": [int(campaigns["observed_conversations"].sum()),
              int(campaigns["orders_created"].sum()),
              int(campaigns["delivered_orders"].sum())],
})
print("Portfolio totals — spend: EGP {:,.0f} | net revenue: EGP {:,.0f}".format(
    campaigns["spend"].sum(), campaigns["net_revenue"].sum()))
alt.Chart(funnel).mark_bar(color="#2E8B57").encode(
    x=alt.X("Count:Q", title="Count"),
    y=alt.Y("Stage:N", sort=["Observed conversations","Orders created","Delivered orders"], title=None),
    tooltip=["Stage:N", alt.Tooltip("Count:Q", format=",")],
).properties(height=140, title="The observed outcome funnel (chats → orders → delivered)")


Portfolio totals — spend: EGP 402,275 | net revenue: EGP 457,574


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [6]:
# Portfolio map: each dot is a campaign. Right = more spend, up = more money back.
d = campaigns.copy()
d["type"] = d["campaign_type"].str.replace("_", " ").str.title()
alt.Chart(d).mark_circle(opacity=0.85, stroke="white", strokeWidth=1.5).encode(
    x=alt.X("spend:Q", title="Ad spend (EGP)"),
    y=alt.Y("net_revenue:Q", title="Observed net revenue (EGP)"),
    size=alt.Size("delivered_orders:Q", title="Delivered orders"),
    color=alt.Color("type:N", title="Campaign type"),
    tooltip=["campaign_name:N", alt.Tooltip("net_roas:Q", title="Net ROAS", format=".2f")],
).properties(height=360, title="Portfolio map — spend vs. money back (dots above the diagonal earned back)")


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Step 3 — "Did each campaign do its job, and what next?" ⚖️

Now the **decision**. For each campaign the code:
1. builds an **evidence pack** (its primary KPI vs. a benchmark, plus guardrails),
2. runs the **rule-based assessor** → a `target_status` and a `next_cycle_action`.

Remember: these are **rules**, not AI — same input always gives the same answer.

In [7]:
from src_2.analytics import build_evidence_packs, DeterministicCampaignAssessor
from src_2.infrastructure.configuration import load_campaign_type_registry, load_budget_policy

registry = load_campaign_type_registry()
policy = load_budget_policy()
cycle_id = f"cycle_{canonical.cycle_start.date()}_{canonical.cycle_end.date()}"

packs = build_evidence_packs(cycle_id, scorecards, registry, quality)
assessor = DeterministicCampaignAssessor()
assessments = [assessor.assess(p, registry.campaign_types[p.campaign_type]) for p in packs]

# One campaign explained in full:
p = packs[0]; a = assessments[0]
print("Campaign     :", p.campaign_name, f"({p.campaign_type.value})")
print("Its job      :", p.business_job)
kpi = p.primary_kpis[0]
print(f"Primary KPI  : {kpi.label} = {kpi.actual}  vs benchmark {kpi.benchmark}  -> {'PASS' if kpi.passed else 'FAIL'}")
print("Decision     :", a.next_cycle_action.value, "|", a.target_status.value)
print("Why (codes)  :", a.reason_codes)


Campaign     : Always-On Premium Acquisition (always_on)
Its job      : Acquire steady sales with stable economics.
Primary KPI  : Net return on ad spend = 0.46755342993386106  vs benchmark 1.205623744075724  -> FAIL
Decision     : do_not_fund | not_achieved
Why (codes)  : ['limited_evidence', 'primary:net_roas:fail', 'guardrail:negative_outcome_rate:fail']


**What to look at:** the decision is traceable — it names the KPI, the benchmark,
and the reason. Here's how that primary KPI compares to its benchmark, and how the
whole portfolio's decisions break down:

In [8]:
rows = [{"Metric": m.label, "Which": "This campaign", "Value": m.actual} for m in p.primary_kpis]
rows += [{"Metric": m.label, "Which": "Benchmark", "Value": m.benchmark} for m in p.primary_kpis]
alt.Chart(pd.DataFrame(rows)).mark_bar().encode(
    x=alt.X("Which:N", title=None), y=alt.Y("Value:Q", title=None),
    color=alt.Color("Which:N", scale=alt.Scale(domain=["This campaign","Benchmark"], range=["#3D8DFF","#B8BCC4"]), legend=None),
    column=alt.Column("Metric:N", title=f"{p.campaign_name}: primary KPI vs. benchmark"),
).properties(height=200)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [9]:
LABELS = {"scale":"Scale","keep_as_test":"Keep as test","do_not_fund":"Do not fund",
          "insufficient_evidence":"Insufficient evidence","data_not_ready":"Data not ready"}
counts = (pd.Series([a.next_cycle_action.value for a in assessments]).map(LABELS)
          .value_counts().rename_axis("Decision").reset_index(name="Campaigns"))
alt.Chart(counts).mark_bar(color="#3D8DFF").encode(
    x=alt.X("Campaigns:Q", title="Number of campaigns", axis=alt.Axis(tickMinStep=1)),
    y=alt.Y("Decision:N", sort="-x", title=None),
    tooltip=["Decision:N","Campaigns:Q"],
).properties(height=180, title="Next-cycle decisions across the portfolio")


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Step 4 — "How should we split next cycle's budget?" 💰

The code sizes a **budget envelope** per campaign type (from past spend), funds only
**eligible** campaigns (blocked ones get zero), and normalizes everything to **100
illustrative units**. It's a *starting point for a human*, not an order to spend.

In [10]:
from src_2.analytics import enrich_scorecards, DeterministicBudgetAllocator

enriched = enrich_scorecards(scorecards, assessments, registry, quality)
budget = DeterministicBudgetAllocator().allocate(
    cycle_id, enriched.campaign, assessments, registry, policy, quality)

assigned = sum(x.budget_units for x in budget.allocations)
print(f"Assigned {assigned:.1f} of {budget.total_budget_units:.0f} units; "
      f"{budget.unallocated_units:.1f} left unallocated (blocked campaigns).")
print("Operational?", budget.operational, "(False = illustrative only, do not auto-execute)")

funded = pd.DataFrame([{"Campaign": x.entity_name, "Budget units": x.budget_units}
                       for x in budget.allocations if x.budget_units > 0]).sort_values("Budget units", ascending=False)
alt.Chart(funded).mark_bar(color="#3D8DFF").encode(
    x=alt.X("Budget units:Q", title="Illustrative budget units (out of 100)"),
    y=alt.Y("Campaign:N", sort="-x", title=None),
    tooltip=["Campaign:N", alt.Tooltip("Budget units:Q", format=".1f")],
).properties(height=320, title="Illustrative next-cycle budget split")


Assigned 62.8 of 100 units; 37.2 left unallocated (blocked campaigns).
Operational? False (False = illustrative only, do not auto-execute)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Step 5 — "Now explain it to me in plain English" 🤖

This is the **only** place an AI is used. It receives the **already-computed evidence**
(numbers + the decision) and writes a readable narrative. It **cannot** change any
number or decision — it only explains them.

The cell runs one real AI call *if a key is configured*; otherwise it shows the
evidence the AI would have explained.

In [11]:
if os.getenv("OPENAI_API_KEY"):
    from src_2.intelligence import OpenAICampaignAnalyst
    insight = OpenAICampaignAnalyst().analyze(packs[0])
    print("AI narrative for:", insight.campaign_id)
    print("Assessment:", insight.target_assessment)
    print("Lesson    :", insight.strategic_lesson)
    print("Next test :", insight.next_controlled_test)
else:
    print("(No API key — showing the evidence the AI would explain, for", packs[0].campaign_name, ")")
    print("decision:", assessments[0].next_cycle_action.value)
    print("primary KPI:", packs[0].primary_kpis[0].label, "=", packs[0].primary_kpis[0].actual,
          "passed:", packs[0].primary_kpis[0].passed)


AI narrative for: 120209876543220001
Assessment: always_on — Acquire steady sales with stable economics. Primary KPI (net_roas) failed: campaign net_roas (entity 120209876543220001) = 0.46755342993386106 vs benchmark 1.205623744075724 (evidence: campaign net_roas). The campaign did produce conversations and delivered orders (observed_conversations 111 and delivered_orders 52 at entity 120209876543220001) but economics remain below acceptable thresholds (evidence referenced below).
Lesson    : The campaign is generating volume and conversations but failing the core economics target: net_roas is below target at the campaign level (entity 120209876543220001). Best evidence of acceptable relative economics and quality comes from the lookalike audience and the default premium creative (entities 120209876543220001::audience::lookalike and 120209876543220001::creative::120209876543230027). Focus optimization on lookalike + proven creative while addressing the elevated negative_outcome_rate in

## Step 6 — "Can we trust what the AI said?" ✅

Because the AI is the only non-deterministic part, a **checker** verifies the
narrative is *faithful* to the numbers — e.g. it doesn't invent figures, doesn't
contradict the decision, and admits when evidence is limited. Here we run those
deterministic checks on a sample narrative (offline, no AI needed).

In [12]:
from src_2.contracts import CampaignInsight, EvidenceReference
from src_2.domain.models import EntityLevel
from src_2.evaluation.consistency import run_consistency_checks, consistency_passed

kpi = packs[0].primary_kpis[0]
sample_narrative = CampaignInsight(
    campaign_id=packs[0].campaign_id,
    target_assessment=f"The primary {kpi.label} came in at {kpi.actual}.",
    supporting_evidence=[EvidenceReference(entity_level=EntityLevel.CAMPAIGN,
        entity_id=packs[0].campaign_id, metric=kpi.metric, actual=kpi.actual, benchmark=kpi.benchmark)],
    strategic_lesson="Judge the campaign by its primary KPI.",
    risks_and_confounders=packs[0].limitations[:1] or ["Observed sample only."],
    evidence_status=packs[0].evidence_status,
)
checks = run_consistency_checks(sample_narrative, packs[0], assessments[0])
for c in checks:
    print(f"[{'PASS' if c.passed else 'FAIL'}] {c.name} ({c.severity}) — {c.detail}")
print("\nNarrative trustworthy?", consistency_passed(checks))


[PASS] evidence_status_match (hard) — matches the deterministic evidence status
[PASS] supporting_evidence_valid (hard) — all references resolve to the evidence pack
[PASS] limitations_disclosed (hard) — evidence limitations are disclosed
[PASS] figures_grounded (warn) — all cited decimal figures trace to the evidence

Narrative trustworthy? True


---
# Part 5 — The whole thing in one call, and how it's built

Everything above is what **one function** does end to end:

```python
from src_2.application import run_completed_cycle
report = run_completed_cycle()   # ingest → scorecards → decisions → budget → AI narrative
```

(We don't run it here because the narrative step makes ~14 AI calls; the app does.)

### How the code is arranged (why it's easy to change)
Each replaceable part is a **"port"** — a socket you can swap:

```
        raw data
           │
   ingestion + analytics        ← fixed rules: clean, score, DECIDE, allocate budget
           │   (evidence + decisions, all reproducible & auditable)
           ▼
   ┌─ ports (swappable sockets) ─────────────────────────────┐
   │  CampaignAssessor   → the funding decision   [rules today]│
   │  BudgetAllocator    → the budget split       [rules today]│
   │  CampaignAnalyst    ┐                                      │
   │  PortfolioSynth.    ├→ the narrative          [AI today]   │
   │  ReportNarrator     ┘                                      │
   └──────────────────────────────────────────────────────────┘
           │
   CompletedCycleReport → shown in the app, then checked by the evaluator
```

- **Decisions default to rules** (safe, auditable). An AI *could* be plugged into the
  same sockets, but money/decision sockets stay rule-based unless deliberately swapped.
- **Narrative defaults to AI** — because explaining is exactly what it's good at.
- The **evaluator** (Step 6) watches the AI's output over time.


---
# Part 6 — Try it yourself

Two ready-to-run apps (from the project root):

```bash
python -m streamlit run app_v2.py      # the stakeholder report (the main product)
python -m streamlit run app_eval.py    # the quality dashboard (checks the AI over time)
```

Want the deep technical version of this walkthrough (worked KPI math, the ports
prototype, evaluation internals)? Open **`src_2_walkthrough.ipynb`**.

### You now know
- **Why** the system exists (turn one messy cycle into fund/keep/stop decisions).
- **The golden rule** — rules decide, AI explains, evidence is treated honestly.
- **The workflow** — clean → score → decide → budget → narrate → check.
- **How the code is shaped** — swappable ports around a deterministic core.
